## Axion-matter couplings at the D-flat Kahler point

Imports the matter-field wavefunctions and the Kahler point `t_locus`, computes the matter Kahler metric
`G` and the coupling `Lambda_i`, and reports the volume-direction axion (a universal check) and the two
diagonalised light axions (2 light + 2 eaten, since `rank{k_a}=2`).  Geometry and volume are evaluated at
`t_locus

In [1]:
import numpy as np, pickle
with open('model2_matter_fields.pkl','rb') as f:
    M = pickle.load(f)
nu_pb = M['nu_pb']; Hvals = M['Hvals']; CLOUD = M['cloud']
SECTORS = M['sectors']; FIELD_MAP = M['field_map']
aux = M['aux']; V_CY = M['V_CY']; t_locus = np.asarray(M['t_locus'])
N_val = len(aux)
print("fields :", list(nu_pb))
print("sectors:", list(SECTORS))
print("t_locus:", t_locus, "  V_CY:", round(V_CY,4), "  N:", N_val)

fields : ['U1', 'U2', 'U3', 'Q1', 'Q2', 'Q3', 'E1', 'E2', 'E3', 'D', 'L']
sectors: ['Q', 'U', 'E', 'D', 'L']
t_locus: [4.         1.         3.         1.33333333]   V_CY: 74.6667   N: 50000


## The two overlap integrals (single cloud, geometry at `t_locus`)

`G_IJ = (1/2V) INT nu_I ^ *(H nu_J)` and `Lambda_iIJ = (1/4V) INT (*omega_i) ^ nu_I ^ (H nu_J)`, each from its own Hodge star.  `g` is the trained metric at `t_locus`, so `J = sum t^i omega_i` is encoded
automatically and the dilation identity `sum t^i Lambda_i = G` holds at `t_locus`.

In [2]:
_e3 = np.zeros((3,3,3))
for _p,_s in [((0,1,2),1),((1,2,0),1),((2,0,1),1),((0,2,1),-1),((2,1,0),-1),((1,0,2),-1)]: _e3[_p]=_s
g = CLOUD['g']; ginv = CLOUD['ginv']; gref = CLOUD['gref']
trgW = np.real(np.einsum('nab,niba->ni', ginv, gref))          # Lambda(omega_i)=tr(g^{-1}W^{(i)})

def sector_integrals(fields):
    nu = np.stack([nu_pb[f] for f in fields], 0)
    n = len(fields); G = np.zeros((n,n),complex); Lam = np.zeros((4,n,n),complex)
    for I in range(n):
        H_I = Hvals[fields[I]]
        for J in range(n):
            Hnu = H_I[:,None]*np.conj(nu[J])
            Qgg = np.einsum('xab,xcd,xe,xf,acf,bde->x', g, g, nu[I], Hnu, _e3, _e3)
            G[I,J] = (1.0/(4*V_CY))*np.mean(aux*Qgg)
            for i in range(4):
                QgW = np.einsum('xab,xcd,xe,xf,acf,bde->x', g, gref[:,i], nu[I], Hnu, _e3, _e3)
                # 0.5 = GS/one-loop vertex 1/(4pi)
                Lam[i,I,J] = 0.5*(-(1.0/(2*V_CY))*np.mean(aux*QgW) + (1.0/(4*V_CY))*np.mean(aux*trgW[:,i]*Qgg))
    G = 0.5*(G+G.conj().T)
    for i in range(4): Lam[i] = 0.5*(Lam[i]+Lam[i].conj().T)
    return G, Lam

results = {}
for r,(fields, blk, univ) in SECTORS.items():
    G, Lam = sector_integrals(fields)
    results[r] = dict(G=G, Lam=Lam, fields=fields, univ=univ)
    print(f"sector {r:2s} {fields}: G {G.shape}{'  (U(3): scalar x3)' if univ else ''}")

sector Q  ['Q1', 'Q2', 'Q3']: G (3, 3)
sector U  ['U1', 'U2', 'U3']: G (3, 3)
sector E  ['E1', 'E2', 'E3']: G (3, 3)
sector D  ['D']: G (1, 1)  (U(3): scalar x3)
sector L  ['L']: G (1, 1)  (U(3): scalar x3)


## Matter Kahler metrics `G` (the normalisations)

In [3]:
def showM(M, ind="    "): print(ind + np.array2string(np.round(M,5), prefix=ind))
def eig(M): return np.round(np.linalg.eigvalsh(0.5*(M+M.conj().T)),6)
for r in results:
    G = results[r]['G']
    print(f"\n  G^{r}  {results[r]['fields']}{'  x3 (U(3))' if results[r]['univ'] else ''}:"); showM(G)
    if G.shape[0] > 1: print(f"    eigs = {eig(G)}")


  G^Q  ['Q1', 'Q2', 'Q3']:
    [[ 0.07357+0.00e+00j -0.00224-2.85e-03j -0.01657+1.00e-05j]
     [-0.00224+2.85e-03j  0.05016+0.00e+00j  0.00047+9.00e-05j]
     [-0.01657-1.00e-05j  0.00047-9.00e-05j  0.07473+0.00e+00j]]
    eigs = [0.049246 0.058302 0.090909]

  G^U  ['U1', 'U2', 'U3']:
    [[ 0.07471+0.00e+00j  0.00146+6.10e-04j -0.01667-3.00e-05j]
     [ 0.00146-6.10e-04j  0.04946+0.00e+00j -0.00066-1.88e-03j]
     [-0.01667+3.00e-05j -0.00066+1.88e-03j  0.07482+0.00e+00j]]
    eigs = [0.049014 0.058472 0.091501]

  G^E  ['E1', 'E2', 'E3']:
    [[ 0.07471+0.00e+00j  0.00146+6.10e-04j -0.01667-3.00e-05j]
     [ 0.00146-6.10e-04j  0.04946+0.00e+00j -0.00066-1.88e-03j]
     [-0.01667+3.00e-05j -0.00066+1.88e-03j  0.07482+0.00e+00j]]
    eigs = [0.049014 0.058472 0.091501]

  G^D  ['D']  x3 (U(3)):
    [[0.32507+0.j]]

  G^L  ['L']  x3 (U(3)):
    [[0.32171+0.j]]


## Axion kinetic metric and the 2 eaten / 2 light directions (at `t_locus`)

`g_ij = (1/4) d_i d_j(-ln V)` at `t_locus`.  `rank{k_a}=2` (since `k1=k2=k3`), so the two eaten
combinations span `{k1,k4}` and the light space is the 2-dim `g`-orthogonal complement (the tangent to
the D-flat cone).  The **volume direction** is the radial `t_locus` (universal by dilation).

In [4]:
def volume(t): return 2.0*(t[0]*t[1]*t[2]+t[0]*t[1]*t[3]+t[0]*t[2]*t[3]+t[1]*t[2]*t[3])
D3 = np.zeros((4,4,4))
for i in range(4):
    for j in range(4):
        for k in range(4):
            if len({i,j,k})==3: D3[i,j,k]=2.0
t = t_locus; V = volume(t)
Vi = np.einsum('ijk,j,k->i', D3, t, t)/2.0; Vij = np.einsum('ijk,k->ij', D3, t)
g_axion = 0.25*(-Vij/V + np.outer(Vi,Vi)/V**2)          # (1/4) dd(-ln V) at t_locus
print("g_axion at t_locus:\n", np.round(g_axion,5))

def g_orth(vecs, metric, prev=None):
    out = list(prev) if prev else []
    for v in vecs:
        v = np.asarray(v, float).copy()
        for u in out: v = v - (v@metric@u)/(u@metric@u)*u
        if np.sqrt(abs(v@metric@v)) > 1e-9: out.append(v/np.sqrt(v@metric@v))
    return out

k1 = np.array([-1.,-1,1,1]); k4 = np.array([1.,2,-3,-1])          # the 2 independent anomalous dirs
eaten = g_orth([k1, k4], g_axion)                                 # 2 eaten (g-orthonormal)
light = g_orth([np.eye(4)[i] for i in range(4)], g_axion, prev=eaten)[len(eaten):]   # 2 light
vol_dir = t / np.sqrt(t @ g_axion @ t)                            # volume/radial direction (unit in g)
print(f"# eaten = {len(eaten)}   # light = {len(light)}   (expect 2 + 2)")
print("volume direction in light space?  |g-overlap with eaten| =",
      np.round([abs(vol_dir@g_axion@e) for e in eaten],6))

g_axion at t_locus:
 [[0.01246 0.00287 0.00032 0.00161]
 [0.00287 0.08163 0.0051  0.02583]
 [0.00032 0.0051  0.02041 0.00287]
 [0.00161 0.02583 0.00287 0.06475]]
# eaten = 2   # light = 2   (expect 2 + 2)
volume direction in light space?  |g-overlap with eaten| = [0. 0.]


## Canonical couplings — volume direction, the two light axions, and the eaten (heavy) axions

`xi = (1/sqrt2) * e^i G^{-1/2}(i Lambda_i) G^{-1/2}`  (`1/sqrt2` = real-axion normalisation).   The **eaten (heavy)** directions `k1,k4` (the anomalous $U(1)$s) give the couplings that live inside the massive vector multiplets — non-universal, and part of the genuine model-dependent output.

In [5]:
def canon(r, direction):
    G = results[r]['G']; Lam = results[r]['Lam']
    ev,U = np.linalg.eigh(0.5*(G+G.conj().T)); Gis = U @ np.diag(1/np.sqrt(ev)) @ U.conj().T
    xi = -1j*(Gis @ (1j*np.einsum('i,iIJ->IJ', direction, Lam)) @ Gis)
    return (1/np.sqrt(2)) * xi                          # 1/sqrt2: canonical REAL axion (not the complex modulus)

def show(M):
    print("        diag =", np.round(np.real(np.diag(M)),4))
    if M.shape[0] > 1: print("        " + np.array2string(np.round(M,4), prefix="        "))

print("="*70); print(" VOLUME-DIRECTION axion  (universal check: prop identity per sector)"); print("="*70)
for r in results:
    print(f"  {r:2s} {results[r]['fields']}:"); show(canon(r, vol_dir))

print("\n" + "="*70); print(" TWO DIAGONALISED LIGHT axions  (the physical output)"); print("="*70)
for r in results:
    print(f"  sector {r:2s} {results[r]['fields']}:")
    for a,e in enumerate(light):
        tag = "  (~volume, universal)" if abs(abs(e@g_axion@vol_dir)-1) < 1e-3 else ""
        print(f"    light[{a}]{tag}:"); show(canon(r, e))

print("\n" + "="*70); print(" HEAVY (eaten) axions  (couplings in the massive anomalous-U(1) vector multiplets)"); print("="*70)
for r in results:
    print(f"  sector {r:2s} {results[r]['fields']}:")
    for a,e in enumerate(eaten):
        print(f"    eaten[{a}]:"); show(canon(r, e))

 VOLUME-DIRECTION axion  (universal check: prop identity per sector)
  Q  ['Q1', 'Q2', 'Q3']:
        diag = [0.4232 0.4148 0.4241]
        [[ 0.4232-0.j     -0.0027-0.0059j  0.0048+0.0009j]
         [-0.0027+0.0059j  0.4148-0.j      0.0006+0.0009j]
         [ 0.0048-0.0009j  0.0006-0.0009j  0.4241-0.j    ]]
  U  ['U1', 'U2', 'U3']:
        diag = [0.4242 0.4147 0.4214]
        [[ 4.242e-01-0.j      3.300e-03+0.0019j  4.600e-03+0.001j ]
         [ 3.300e-03-0.0019j  4.147e-01-0.j     -1.000e-04-0.0046j]
         [ 4.600e-03-0.001j  -1.000e-04+0.0046j  4.214e-01-0.j    ]]
  E  ['E1', 'E2', 'E3']:
        diag = [0.4242 0.4147 0.4214]
        [[ 4.242e-01-0.j      3.300e-03+0.0019j  4.600e-03+0.001j ]
         [ 3.300e-03-0.0019j  4.147e-01-0.j     -1.000e-04-0.0046j]
         [ 4.600e-03-0.001j  -1.000e-04+0.0046j  4.214e-01-0.j    ]]
  D  ['D']:
        diag = [0.4541]
  L  ['L']:
        diag = [0.4526]

 TWO DIAGONALISED LIGHT axions  (the physical output)
  sector Q  ['Q1', 'Q2', 'Q